# Aurora visibility — look up a location

In [3]:
import ipywidgets as widgets
from IPython.display import display

from predict_core import predict_aurora_probability

In [ ]:
PRESET_LOCATIONS = {
    "Tromsø, Norway": (69.6492, 18.9553),
    "Fairbanks, Alaska": (64.8378, -147.7164),
    "Reykjavik, Iceland": (64.1466, -21.9426),
    "Yellowknife, Canada": (62.4540, -114.3718),
    "Custom (use lat/lon below)": None,
}

location_dropdown = widgets.Dropdown(options=list(PRESET_LOCATIONS.keys()), value="Tromsø, Norway", description="Location:", style={"description_width": "initial"})
lat_input = widgets.FloatText(value=69.6492, description="Latitude:", style={"description_width": "initial"})
lon_input = widgets.FloatText(value=18.9553, description="Longitude:", style={"description_width": "initial"})
predict_button = widgets.Button(description="Get live prediction", button_style="success")
output_area = widgets.Output()


def _on_location_change(change):
    preset = PRESET_LOCATIONS[change["new"]]
    if preset is not None:
        lat_input.value, lon_input.value = preset


location_dropdown.observe(_on_location_change, names="value")

In [5]:
def _format_result(result):
    prob = result["probability"]
    lines = [f"<h3>P(aurora visible) ≈ {prob:.0%}</h3>"]
    if result["low_confidence"]:
        lines.append(f"<p style='color:#b8860b;'>⚠️ {result['low_confidence_reason']}</p>")
    inp = result["inputs"]
    if inp is not None:
        lines.append(
            "<table style='font-size:0.9em;'>"
            f"<tr><td>mlat</td><td>{result['mlat']:.1f}°</td></tr>"
            f"<tr><td>kp</td><td>{inp['kp']:.2f}</td></tr>"
            f"<tr><td>bz_gsm</td><td>{inp['bz_gsm']:.2f} nT</td></tr>"
            f"<tr><td>solar_wind_speed</td><td>{inp['solar_wind_speed']:.0f} km/s</td></tr>"
            f"<tr><td>cloud_cover</td><td>{inp['cloud_cover']:.0f}%</td></tr>"
            f"<tr><td>sun_elevation_utc</td><td>{inp['sun_elevation_utc']:.1f}°</td></tr>"
            f"<tr><td>moon_darkness</td><td>{inp['moon_darkness']:.2f}</td></tr>"
            "</table>"
        )
    return "".join(lines)


def _on_predict_click(_button):
    output_area.clear_output()
    with output_area:
        print("Fetching live data...")
        try:
            result = predict_aurora_probability(lat=lat_input.value, lon=lon_input.value)
        except Exception as exc:
            output_area.clear_output()
            print(f"Prediction failed: {exc}")
            return
        output_area.clear_output()
        display(widgets.HTML(_format_result(result)))


predict_button.on_click(_on_predict_click)

display(widgets.VBox([location_dropdown, widgets.HBox([lat_input, lon_input]), predict_button, output_area]))